# RepairScopeX R08 — Independent Held-out Decoding-Budget Confirmation

**Updated after the completed R06 budget-control and R07 prompt-sensitivity results.**

- 48 **new** public benchmark test problems (24 GSM8K + 24 SVAMP), excluded by ID and normalized question SHA-256 from all 96 R02/R03 problems, and consequently from the nested R06/R07 cohorts.
- Four source/intervention conditions per problem: **source128, source256, finalize128 after source128, regenerate256**. Exactly 192 planned ALoDLM model calls. Preserve/no intervention is the source output (zero extra cost).
- Primary outcome: **strict numeric final-answer correctness** using the historical R06-compatible rubric. Other outputs are classified *ambiguous*, not automatically mathematically wrong.
- Paired exact tests, bootstrap intervals, completion rates, mean total forward-token passes and latencies. A separate blinded review queue is created; it is **not automatically human-adjudicated**.
- Model/dataset revisions pinned, case selection frozen **before the first call**, immutable per-trial JSON checkpoint on Google Drive, 12 new calls or 27 minutes per session.

R08 is a *follow-up replication* designed after observing R06, not a preregistered test of the original RepairScopeX hypothesis. A 128+128 two-call pipeline has the same nominal generated-token ceiling as one 256-token call, **but not identical inference cost** because of repeated prompt processing and different stopping conditions. No paid API keys.


In [ ]:
import os,sys,subprocess
os.environ['HF_HUB_DISABLE_XET']='1';os.environ['TOKENIZERS_PARALLELISM']='false'
subprocess.run([sys.executable,'-m','pip','install','-q',
  'transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1',
  'datasets>=3.6,<5','pandas>=2','scipy>=1.12','safetensors>=0.6','PyYAML>=6'],check=True)
from google.colab import drive
drive.mount('/content/drive')
import torch
if not torch.cuda.is_available():raise RuntimeError('T4-class GPU required; no CPU fake results')


In [ ]:
import os,sys,json,time,random,hashlib,math,re,zipfile,gc,subprocess
from pathlib import Path
from collections import Counter,defaultdict
import numpy as np,pandas as pd
from scipy.stats import binomtest
from datasets import load_dataset
from huggingface_hub import snapshot_download

STUDY='RSX_R08_ALoDLM_Fresh_Budget_Confirmation'
ROOT=Path('/content/drive/MyDrive/RepairScopeX/R08_Fresh_Budget_Confirmation')
RECS=ROOT/'records';RECS.mkdir(parents=True,exist_ok=True)
DATASET_REVISIONS={'ChilleD/SVAMP': '5e0bf1e5e7c0e9c4bc39180d224f41f3f801b7ef', 'openai/gsm8k': '740312add88f781978c0658806c59bc2815b9866'}
EXCLUDED_R02_IDS=set(['gsm8k:003c5304d9aeda0f', 'gsm8k:0172bc0c9804157b', 'gsm8k:039b584a1bcb7af9', 'gsm8k:03c263632e80397e', 'gsm8k:04652af213b74506', 'gsm8k:0a90137d878c4667', 'gsm8k:0fd5beca3ad36df1', 'gsm8k:13d10b8bfcf30de5', 'gsm8k:13e140ab86daaa95', 'gsm8k:15e94c36279f5ef4', 'gsm8k:166c80c7c0564d74', 'gsm8k:173e1d62ebee6eda', 'gsm8k:213b8d6329ee91c7', 'gsm8k:23fdf08ce87602ed', 'gsm8k:245cec2cc9f80779', 'gsm8k:270649c36e2edec1', 'gsm8k:2c33011136b85c88', 'gsm8k:2e203ecfb155ce3e', 'gsm8k:316eac029fc34d0d', 'gsm8k:3d994447a15548bb', 'gsm8k:3e64aca4939f160d', 'gsm8k:403c14d7e3268a34', 'gsm8k:40c374ea90760ff2', 'gsm8k:44f3552fcc34a396', 'gsm8k:477ccaf5ccaf2021', 'gsm8k:4b4052802bfb783f', 'gsm8k:4e332330d73d72eb', 'gsm8k:528398375ba806bc', 'gsm8k:55091194deedd01e', 'gsm8k:7207f9cd655e1e79', 'gsm8k:851e6439f44e55ee', 'gsm8k:95e5f2d591336b3e', 'gsm8k:a560fa2733b9bef0', 'gsm8k:a6d3a04eaa27136f', 'gsm8k:a7f615dad911a47a', 'gsm8k:a80947ed1965b3e7', 'gsm8k:a9e516a6f16137b1', 'gsm8k:bdd8988e8c52f0f4', 'gsm8k:bfd3566a5a978c0c', 'gsm8k:c46455f66de332c0', 'gsm8k:c5a5231d32ac6318', 'gsm8k:d09c58e99911f2b9', 'gsm8k:d28d879c9589742f', 'gsm8k:dd6cf24849d2716e', 'gsm8k:dde46487272fd041', 'gsm8k:dfbf7d8a56e41804', 'gsm8k:e5dbfc5b4e41fb86', 'gsm8k:fb0a24a6ed729984', 'svamp:06b3927ad433ee83', 'svamp:0f45a2efbcf48ac2', 'svamp:0fd98afbe13958ee', 'svamp:15353453c8460c69', 'svamp:17ea26097d088c3c', 'svamp:20e778bdb4a5b132', 'svamp:2e35fcf3fc209aab', 'svamp:32373e088b00fa44', 'svamp:33a06fedfa28186e', 'svamp:33ee6721af81c3d3', 'svamp:34cf8def215d61b7', 'svamp:3676dc92011399df', 'svamp:3bfc27618f694232', 'svamp:3c4e70a5039a316f', 'svamp:3d1975b379205278', 'svamp:40db943f8b10441a', 'svamp:4aa8dc056fcd7a1c', 'svamp:518a8a06f28c5f1d', 'svamp:573fefef021c5e2f', 'svamp:57e72d49395d8783', 'svamp:58d29803fe0f08a3', 'svamp:59b1c90201b15381', 'svamp:5f71f598acd9e3c0', 'svamp:7514b1a3ea2a7d5b', 'svamp:773159d2e8417a2b', 'svamp:7bd5a38778820c8f', 'svamp:8314cf2cbe3a3f64', 'svamp:874cce8c39a98f00', 'svamp:8fe6da589fcbcdd2', 'svamp:9830ece26ca3600a', 'svamp:a4ec25b4e22d4e1c', 'svamp:aac9408671577594', 'svamp:b24c670664615ba6', 'svamp:b516be6864b673e1', 'svamp:b5d9e1a516407168', 'svamp:b92820e4f2e38e9d', 'svamp:bdd97190e7a2ebd3', 'svamp:be2629589f895910', 'svamp:c38c1419354341cc', 'svamp:c392a7160b11839c', 'svamp:d0774a2955f4a82f', 'svamp:dd4dc9c5089be1bb', 'svamp:e69377e18aee2bae', 'svamp:e7fe98b21b9f118d', 'svamp:eadc701a88170b4f', 'svamp:f26571f481fbbf10', 'svamp:f90d6f5c10eedf19', 'svamp:f932111878f7391a'])
EXCLUDED_R02_PROMPT_HASHES=set(['003c5304d9aeda0ff30e926faf27e7c05d3f50f4092b2f11e6230f8f386a342e', '0172bc0c9804157b5970e65921f8da5f7310e035b257e86f66ddcedef2dd8ce9', '039b584a1bcb7af96a16461fc5a6863404331c03187ec56d1135f1eb26b8f1c3', '03c263632e80397e8466508048e2f8309a5d6a2a0ff4c84df6359123bf2e9714', '04652af213b74506abd3f15174298a9bc0e0ac09aed9eed6b1f1edad55136824', '06b3927ad433ee83233aa3b9ec4e7cf1bd6e7b54b4cd2f85f3d363dd12a274c2', '0a90137d878c466752ea670315566fc7c486b9a1c8c941728aa090500268b88a', '0f45a2efbcf48ac267ce05e8616534e788b78e8f63962b51456447e0da11210d', '0fd5beca3ad36df10907617d0ae85d7e9857e29010070a97c95beefdb8484243', '0fd98afbe13958ee4c22943c68e5b9144a4c2f0b3b44f21cbc7fdb9166ed2e03', '13d10b8bfcf30de566fc28f6fc1cc547a3241103c9d9304facc1e4b22c4c58a9', '13e140ab86daaa95907a1336e9956e8b9ba84bb7d22a4e830e4b5b91c577b6c1', '15353453c8460c69f6251a2c107fb8e593f5378aca9734e5cbe15b8742027319', '15e94c36279f5ef449d222b30bb3d5fcc769b8598be32965e08594bfe484e37c', '166c80c7c0564d74494e4b196dcd481ea905a73b98ebf210377c95b7275ab955', '173e1d62ebee6edae8489a337e8278c273017bc478a2df85b70e488096d1d89a', '17ea26097d088c3c82ed8309c9cc993d97c5007a5623aa074a0213f87b87ae52', '20e778bdb4a5b1324bdd6748cd8a6026a3c699e58ec7cf64ec896760ff625706', '213b8d6329ee91c7f3bd3ee7bce5818db01b08b0d107b26c757745cf26e26f21', '23fdf08ce87602ede28ef4240e103f5089e507cb20da5a873e82fb1588dda333', '245cec2cc9f80779aeb56cb55a31678295eafa936b4d85ef46419fc12918382e', '270649c36e2edec1ceb7deb2a07a6ea1964f52708feec3b1853f110cff0839db', '2c33011136b85c88d39be60f1352d69d60a4c372ba0684b6bff4574c081676dd', '2e203ecfb155ce3e67c787a3b80cb06953260d5b4373623abc67d5aa14c07490', '2e35fcf3fc209aabf810a1990e9d796df374bac7eeba122e4c3c68657ea32010', '316eac029fc34d0d5a90a00547c38e110cdb2120010adbe02e493829714c1c02', '32373e088b00fa4447184674ed79a4341d83a3096f6ef6efbf23db3da4e32362', '33a06fedfa28186eb375e31e878a33cea9b3eb406341a4271ce4a1843aba50f6', '33ee6721af81c3d3edc136cc607e918c23d1ba7c6a8f4eb1b38604f087ea4338', '34cf8def215d61b78748b9f981280e5f0cf6f5aaff2af341aa2acaaad29e121d', '3676dc92011399df469422a2959f0bf153448fdcb6c37319f249a7f488ebff05', '3bfc27618f6942323f9770bd8454d981986110180f10c75afdc9925041b0ef35', '3c4e70a5039a316faabb4a32b0a84aa91c4f60d259c122a6cd9f4e45a72d04ff', '3d1975b379205278f898c7fdf84f089b18dfbb331e8a59fd62d9232ad7a0c9c1', '3d994447a15548bb0e94920d9adc01633b79d859fcdeb08783c42b92fd642f83', '3e64aca4939f160dd586bae5c796b69f11d8d2c0165a3276d8e4a359ca98d479', '403c14d7e3268a34e7db02d081f0111ca60985533e231e07ab8787d3b58d03c3', '40c374ea90760ff260995cbb7e2149ee9fc7962c732a5708a2154596540e8e59', '40db943f8b10441a17fb3ac54cfe5927f013183adb4a437ebc46ef456b7841ed', '44f3552fcc34a3969d8d75e860e456093bf8cac8c8ec13c4335fe66c7f6f39e7', '477ccaf5ccaf202145cbe574456faf5de3dd15eaa57124ec964d4c1e6b74233b', '4aa8dc056fcd7a1c8374cc86dc628fa7fe0c3e8efbc340a5eaccd15ea0e4387f', '4b4052802bfb783f728a7869d6847d31b3a0e5faa7daf496781ffc87e424083b', '4e332330d73d72eb50e5e8d1e7a977e31b554f94be78b92826b0c60f8aa30d5a', '518a8a06f28c5f1dd37f1664649c2ddb85d4edeada73d6189cf775e22b7cef39', '528398375ba806bcc8adb71d541b242984ca34de54148424c6a17fd4a3b61f2a', '55091194deedd01e75ab16a799766d76ad0ecc478b159e0a72c51a19eba1dcbc', '573fefef021c5e2f88064fafa6d6a54247536a55747e54aba9528f3b2bf2cea7', '57e72d49395d878377c8158956762e9ac18a163253aed3c7ae3948540a3af102', '58d29803fe0f08a393aadba9901a30080d1ac69be83e17de0f25b7e3317e4eb8', '59b1c90201b153812775e262ad5b29e3ad888fe38995f852018f1daa627afc94', '5f71f598acd9e3c06531acb72bd0600d3ae5b60a1f50717899c1cd69b087966a', '7207f9cd655e1e79949adb7eb8a3279440c3b142f56b2fc841d933e642704242', '7514b1a3ea2a7d5b2881193bcd6b486c1cef4c56c0ba4d5b12a98cb5b04d03b1', '773159d2e8417a2b6dd1819bd534be51c6ae3fa91fcd5b2bbd4ceb1d3b5e0eab', '7bd5a38778820c8f53e3e98900ef5b0bceeaf2de265f53d471fef37910f62620', '8314cf2cbe3a3f64bd731afebe4ff62bafb7d3f32214faa4e90de130c0716676', '851e6439f44e55eecd8d0ae494751c927b1fe964507c75e5709fd09fb2dab402', '874cce8c39a98f00c8d49a6342a971d8dd19eb1745d9d163936e0bc26d6c5315', '8fe6da589fcbcdd2c567ee96ede64d0f8ef493c42af178aa69ca7f237c2d9e18', '95e5f2d591336b3e38b7363732cb3c16d8dd43ea54ba216f78e99ddb0042bc50', '9830ece26ca3600a817663e457eee4a51f4e7c35b7e9fceb1982e3542cf0c077', 'a4ec25b4e22d4e1c3da7f62c5475e5ca527ea97dfba5de6622b1fb8acb676ff6', 'a560fa2733b9bef0401c069cca4435f95cf5ed2fc57807e04aeb47cdba49c582', 'a6d3a04eaa27136fb27acc182076bcf9f46e5459b872f07da3d4620f9f884a71', 'a7f615dad911a47af2d48a0c225f9e437f79d3043ad2969e3d981c31fcda8886', 'a80947ed1965b3e7159b571341dbe2f803beeb5fdd360d66a38c40afc70baa8c', 'a9e516a6f16137b1875125c31f27fc4a0954dedb6f4e749bfa9544c4f01ae100', 'aac94086715775946bf1f02fd5f5b195a23375675db90018d36c4ad86ee384af', 'b24c670664615ba6160fa0420960dd0aedaff3c948da16d0a23eee296ebfaa9b', 'b516be6864b673e1d1628ef24abdfbfd4455a3f6c2604478422cf37399b703fc', 'b5d9e1a5164071686e7c02a02d1507f48174583629f5a92644e5f0814c54169b', 'b92820e4f2e38e9dd2f331a819119f098cb8b498a6ca7238265279a9d20551d3', 'bdd8988e8c52f0f4dbae8ceaa9ff30be454fb592f943d4c205ed21f0f0f44c75', 'bdd97190e7a2ebd36f4262be2a6e61caf6225079729629388171c70e12b13f96', 'be2629589f895910f29063867a8fa40562c429e54bc78d03bfb8560ca26af680', 'bfd3566a5a978c0c7b052db32e6d037d44175c41da0f38f5cfec0ccc5e77b179', 'c38c1419354341ccbe83b98d46eec75c2f894645d1f1857c8b979446a75d5312', 'c392a7160b11839c74d436ab33fca2066cd932ee1c2ae34615421bbfb509d02c', 'c46455f66de332c03de5ac89a8660ac1d31269cc6df97ea4258e056a29446865', 'c5a5231d32ac6318870984b6980ea00b437486c441f6f9eaa38b78ad2096b097', 'd0774a2955f4a82f0407354a9967bf9ee57248af3bcd2baa168248fcd2755ba5', 'd09c58e99911f2b94d31ca7253b5e3bae34d4e3f264c562ce5e7daaa53f3b8aa', 'd28d879c9589742f3cb5852e22b1e0b0ca8399cbc97183339662a347945325fb', 'dd4dc9c5089be1bbdbea915d7f5c91f5921e0e7ab796d96320ce351a119f5d81', 'dd6cf24849d2716eb4ea7348e13e02909f84bba9dfe3a025c811a0858b6c2630', 'dde46487272fd041e5941aececdb7c932a4d3663f4e28ada3057e9169dbc4e6f', 'dfbf7d8a56e41804a2dbcea3322b54b270f2770c57ae5e1c0f54ec2510c952ff', 'e5dbfc5b4e41fb868cc102dcdef941ae865bb02687210fd701b7c23ba410ba28', 'e69377e18aee2bae59a38c9dff83f84cd4bfd4db57058b0a7cdf7eafb471f590', 'e7fe98b21b9f118dbd526b4bb4ff4858ddec021823790b19a7e01ce516feba88', 'eadc701a88170b4fb3b9c8b325bfe09fd6a5b633c78e1ddb11081183c1001d7e', 'f26571f481fbbf108ef42883f42f56a03f875ec7f98bd93524ee6647e0d84073', 'f90d6f5c10eedf19efa88285b2e8da4f2663331501a839ab23e6d5cb67ca05e4', 'f932111878f7391a56070e84a6c5b683d1b67740f294110a2b70fd31f8f2914f', 'fb0a24a6ed729984b2301db48c850b6bc957f4fd51f5872f28901fa9cbdede3c'])
R02_MANIFEST_SHA256='659c32eefa5540c959ffe21a639e238b7998e5e21b63870831b60016cb61cdaf'
assert len(EXCLUDED_R02_IDS)==96 and len(EXCLUDED_R02_PROMPT_HASHES)==96

def sha(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def sha_bytes(s):return hashlib.sha256(s.encode()).hexdigest()
def atomic_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        if json.loads(path.read_text())!=obj:raise RuntimeError('IMMUTABLE FILE MISMATCH '+str(path))
    else:
        temp=path.with_name(path.name+'.writing')
        temp.write_text(json.dumps(obj,sort_keys=True,ensure_ascii=False,indent=2))
        os.replace(temp,path)

# Conservative historical-primary parser. Missing/malformed/conﬂicting final markers => ambiguous.
# R05 human labels are a separate endpoint, never silently substituted.
def grade_explicit(output,target):
    t=str(output)
    forms=[r'(?im)^\s*FINAL_ANSWER\s*:\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*[.!]?\s*$',
           r'\\boxed\s*\{\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}',
           r'(?im)^\s*(?:final )?answer\s*:\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*[.!]?\s*$']
    numbers=[]
    for form in forms:
        numbers += [float(v) for v in re.findall(form,t)]
    if not numbers or any(not math.isclose(numbers[0],n,abs_tol=1e-9,rel_tol=1e-9) for n in numbers):
        return {'label':'ambiguous','value':None,'type':'incomplete_or_conflicting'}
    truth=float(target)
    return {'label':'correct' if math.isclose(numbers[0],truth,abs_tol=1e-8,rel_tol=1e-8) else 'wrong',
            'value':numbers[0],'type':'complete_numeric_final'}
assert grade_explicit('FINAL_ANSWER: 17',17)['label']=='correct'
assert grade_explicit('FINAL_ANSWER: 19',17)['label']=='wrong'
assert grade_explicit('Reasoning 17',17)['label']=='ambiguous'
assert grade_explicit('\\boxed{17}',17)['label']=='correct'

# Freeze a holdout chosen from the *remaining* public test pool, not from earlier R02/03/06/07 questions.
# Dataset SHA/revisions and selection algorithm are embedded here, independent of outcome labels.
DATA_PATH=ROOT/'R08_FROZEN_CASES.json'
if DATA_PATH.exists():
    sealed=json.loads(DATA_PATH.read_text())
    assert sealed['selection']=='next24_SHA_rank_per_dataset_excluding_R02_96_v1'
    assert sealed['revisions']==DATASET_REVISIONS
    assert sealed['r02_manifest_sha256']==R02_MANIFEST_SHA256
    CASES=sealed['cases']
else:
    CASES=[]
    for repo,config,prefix in [('openai/gsm8k','main','gsm8k'),('ChilleD/SVAMP',None,'svamp')]:
        ds=load_dataset(repo,config,split='test',revision=DATASET_REVISIONS[repo])
        eligible=[]
        for idx,row in enumerate(ds):
            if prefix=='gsm8k':
                q=str(row['question'])
                m=re.search(r'####\s*([+-]?[\d,]+(?:\.\d+)?)',str(row['answer']))
                value=float(m.group(1).replace(',','')) if m else None
            else:
                q=(str(row.get('Body',''))+' '+str(row.get('Question',''))).strip()
                try:value=float(str(row.get('Answer')).replace(',',''))
                except (ValueError,TypeError):value=None
            if not q or value is None or not math.isfinite(value):continue
            pid=prefix+':'+sha_bytes(q)[:16]
            prompt_fingerprint=sha_bytes(q.strip())
            if pid in EXCLUDED_R02_IDS or prompt_fingerprint in EXCLUDED_R02_PROMPT_HASHES:continue
            eligible.append({'id':pid,'dataset':prefix,'prompt':q,'target':value,'original_index':idx,
                             'question_sha256':prompt_fingerprint})
        # Same deterministic SHA rank as R02; only the following 24 unselected examples.
        eligible=sorted(eligible,key=lambda x:sha(x['id']))
        assert len(eligible)>=24,(repo,len(eligible))
        CASES+=eligible[:24]
    sealed={'study':STUDY,'selection':'next24_SHA_rank_per_dataset_excluding_R02_96_v1',
            'revisions':DATASET_REVISIONS,'r02_manifest_sha256':R02_MANIFEST_SHA256,
            'excluded_case_ids_sha256':sha(sorted(EXCLUDED_R02_IDS)),
            'cases':CASES}
    atomic_json(DATA_PATH,sealed)
assert len(CASES)==48 and len({c['id'] for c in CASES})==48
assert Counter(c['dataset'] for c in CASES)=={'gsm8k':24,'svamp':24}
assert all(c['id'] not in EXCLUDED_R02_IDS and c['question_sha256'] not in EXCLUDED_R02_PROMPT_HASHES for c in CASES)
assert sealed['excluded_case_ids_sha256']==sha(sorted(EXCLUDED_R02_IDS))
BANK={c['id']:c for c in CASES}
print('R08 frozen new problems:',len(CASES),'manifest:',sha(sealed),'prior R02 excluded:',len(EXCLUDED_R02_IDS))


In [ ]:
from types import SimpleNamespace
from huggingface_hub import snapshot_download
REPO_DIR=Path('/content/ALoDLM_Q1_Final')
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',str(REPO_DIR)],check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR),'--no-deps'],check=True)
sys.path.insert(0,str(REPO_DIR))
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_ID='amazon/ALoDLM-1.7B';MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
MODEL_PATH=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
args=SimpleNamespace(model=MODEL_PATH,model_config=None,device='cuda',precision='fp32')
decoder=load_decoder(args);tokenizer=decoder.tokenizer
assert next(decoder.model.parameters()).dtype==torch.float32

def model_generate(prompt,budget,seed):
    random.seed(seed);np.random.seed(seed%(2**32-1));torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    ids=list(tokenizer.apply_chat_template([{'role':'user','content':prompt}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
    config=DecodeConfig(q=0.4,tau=0.4,mode='left1',window_size=16,max_new_tokens=budget,position_penalty=None)
    config.validate()
    stamp=time.monotonic()
    result=decoder.generate(ids,config)
    text=tokenizer.decode(list(map(int,result['token_ids'])),skip_special_tokens=True).strip()
    return {'output':text,'nfe':int(result.get('outer_steps',0)),
        'loop_token_passes':int(result.get('loop_token_passes',0)),
        'generated_tokens':int(result.get('generated_tokens',0)),
        'stop_reason':str(result.get('stop_reason','unknown')),
        'wall_seconds':round(time.monotonic()-stamp,3)}


In [ ]:
PHASES=('source128','source256','finalize128','regenerate256')
BUDGETS={'source128':128,'source256':256,'finalize128':128,'regenerate256':256}
SETTINGS={'architecture':MODEL_ID,'model_revision':MODEL_REV,'code_commit':CODE_COMMIT,
          'decoder':'left1','window':16,'q':0.4,'tau':0.4,'precision':'fp32','budgets':BUDGETS,
          'source_prompt':'R06 short reasoning and explicit FINAL_ANSWER',
          'source_finalization':'R06 condition on 128-token original output, ask for final answer',
          'regen_prompt':'R06 independent regeneration prompt with 256-token allowance',
          'evaluation':'strict_v1 from R06; ambiguous != wrong',
          'total_pipeline_cost':'source128 + finalize128: token passes, NFE and time summed'}
PLAN=[[c['id'],phase] for c in CASES for phase in PHASES]
assert len(PLAN)==192
PROTOCOL={'study':STUDY,'source_manifest_sha256':sha(sealed),'cases_sha256':sha(CASES),
          'settings':SETTINGS,'plan':PLAN}
PROTOCOL_HASH=sha(PROTOCOL)
atomic_json(ROOT/'protocol.json',{'protocol':PROTOCOL,'protocol_hash':PROTOCOL_HASH})
atomic_json(ROOT/'cases.json',CASES)

def trial_id(pid,phase):return STUDY+'|'+pid+'|'+phase
def read_record(pid,phase):
    tid=trial_id(pid,phase);path=RECS/(sha_bytes(tid)+'.json')
    if not path.exists():return None
    value=json.loads(path.read_text())
    assert value['trial_id']==tid and value['protocol_hash']==PROTOCOL_HASH
    return value
def save_record(r):
    assert r['protocol_hash']==PROTOCOL_HASH
    atomic_json(RECS/(sha_bytes(r['trial_id'])+'.json'),r)
def records():
    x=[json.loads(p.read_text()) for p in RECS.glob('*.json')]
    assert len(x)==len({r['trial_id'] for r in x})
    assert all(r['protocol_hash']==PROTOCOL_HASH for r in x)
    return x

def prompt_for(c,phase):
    q=c['prompt']
    original=q+'\nGive a brief solution and finish with FINAL_ANSWER: <number>.'
    if phase in ('source128','source256'):return original
    if phase=='regenerate256':return q+'\nSolve independently from scratch; check the numerical result. End with FINAL_ANSWER: <number>.'
    if phase=='finalize128':
        previous=read_record(c['id'],'source128')
        if previous is None:raise RuntimeError('source128 record not available '+c['id'])
        return q+'\nA previous response was:\n'+str(previous['output'])[:1300]+'\nCheck this response and give the correct final number as FINAL_ANSWER: <number>.'
    raise ValueError(phase)

def export_snapshot():
    path=Path('/content')/(STUDY+'_Snapshot.zip')
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in ROOT.rglob('*'):
            if p.is_file() and not p.name.endswith('.writing'):z.write(p,p.relative_to(ROOT))
    print('Saved',path,'size bytes',path.stat().st_size)
    try:
        from google.colab import files
        files.download(str(path))
    except Exception as e:print('Download manually:',path,repr(e))
    return path

print('R08 protocol hash',PROTOCOL_HASH,'existing',len(records()),'/',len(PLAN))


In [ ]:
SESSION_MAX_NEW=12
SESSION_MAX_MINUTES=27
started=time.monotonic();new=0;error=None
for pid,phase in PLAN:
    if new>=SESSION_MAX_NEW or time.monotonic()-started>=SESSION_MAX_MINUTES*60:break
    if read_record(pid,phase) is not None:continue
    c=BANK[pid];tid=trial_id(pid,phase)
    try:
        prompt=prompt_for(c,phase)
        seed=20261025 + int(sha_bytes(pid)[:8],16)%1_000_000
        result=model_generate(prompt,BUDGETS[phase],seed)
        r={'trial_id':tid,'problem_id':pid,'dataset':c['dataset'],'phase':phase,
           'protocol_hash':PROTOCOL_HASH,'seed':seed,'prompt_sha256':sha_bytes(prompt),
           'grade':grade_explicit(result['output'],c['target']), 'reference':c['target'],**result}
        save_record(r);new+=1
        print('SAVE',new,'/',SESSION_MAX_NEW,phase,pid,r['grade']['label'],flush=True)
    except Exception as exc:
        error={'trial_id':tid,'error':repr(exc)}
        (ROOT/'last_error.json').write_text(json.dumps(error,indent=2))
        if 'out of memory' in str(exc).lower():torch.cuda.empty_cache()
        print('STOP on execution error; prior records preserved:',repr(exc));break
print('Session new calls',new,'total completed',len(records()),'/',len(PLAN))
if error:print('Next blocked trial:',error['trial_id'])


In [ ]:
lookup={(r['problem_id'],r['phase']):r for r in records()}
complete=[c for c in CASES if all((c['id'],p) in lookup for p in PHASES)]
rows=[]
for c in complete:
    x={'problem_id':c['id'],'dataset':c['dataset']}
    for ph in PHASES:
        r=lookup[(c['id'],ph)]
        x[ph+'_correct']=int(r['grade']['label']=='correct')
        x[ph+'_ambiguous']=int(r['grade']['label']=='ambiguous')
        x[ph+'_wrong']=int(r['grade']['label']=='wrong')
        x[ph+'_wall_seconds']=float(r['wall_seconds'])
        x[ph+'_nfe']=int(r['nfe'])
        x[ph+'_loop_passes']=int(r['loop_token_passes'])
        x[ph+'_stop_reason']=r['stop_reason']
    base=lookup[(c['id'],'source128')]
    f=lookup[(c['id'],'finalize128')]
    x['finalize_pipeline_total_passes']=base['loop_token_passes']+f['loop_token_passes']
    x['finalize_pipeline_total_nfe']=base['nfe']+f['nfe']
    x['finalize_pipeline_total_seconds']=base['wall_seconds']+f['wall_seconds']
    x['source128_is_prefix_of_source256']=lookup[(c['id'],'source256')]['output'].startswith(base['output'])
    rows.append(x)
df=pd.DataFrame(rows)
if not df.empty:
    df.to_csv(ROOT/'paired_complete_cases.csv',index=False)
    print('Complete independent problems:',len(df),'/',len(CASES))
    print(df.groupby('dataset')[[p+'_correct' for p in PHASES]].agg(['sum','count']))
else:print('No fully paired problems yet; old records preserved.')

# Only write a final test report when ALL 48 cases and 192 unique records are present.
if len(complete)==len(CASES):
    assert len(records())==len(PLAN)==192
    pairs=[('source128','source256'),('source256','finalize128'),
           ('source256','regenerate256'),('finalize128','regenerate256')]
    results={}
    for a,b in pairs:
        x=df[a+'_correct'].to_numpy(dtype=int);y=df[b+'_correct'].to_numpy(dtype=int)
        wins=int(((x==0)&(y==1)).sum());loses=int(((x==1)&(y==0)).sum())
        dif=y-x
        rng=np.random.default_rng(2026102508)
        ci=np.quantile([np.mean(dif[rng.integers(len(dif),size=len(dif))]) for _ in range(10000)], [.025,.975])
        results[b+'_vs_'+a]={'n':int(len(df)),'correct_reference':int(x.sum()),'correct_comparison':int(y.sum()),
            'gain':wins,'harm':loses,'paired_exact_p':float(binomtest(min(wins,loses),wins+loses,.5).pvalue) if wins+loses else 1.0,
            'absolute_accuracy_difference':float(np.mean(dif)),'paired_bootstrap_95_ci':ci.tolist()}
    for ph in PHASES:
        results['cost_'+ph]={'mean_forward_token_passes':float(df[ph+'_loop_passes'].mean()),
          'mean_nfe':float(df[ph+'_nfe'].mean()),'mean_wall_seconds':float(df[ph+'_wall_seconds'].mean()),
          'ambiguous':int(df[ph+'_ambiguous'].sum()),'wrong':int(df[ph+'_wrong'].sum())}
    results['cost_finalize_2_call_pipeline']={
       'mean_forward_token_passes':float(df['finalize_pipeline_total_passes'].mean()),
       'mean_nfe':float(df['finalize_pipeline_total_nfe'].mean()),
       'mean_wall_seconds':float(df['finalize_pipeline_total_seconds'].mean())}
    results['source128_prefix_of_source256_count']=int(df['source128_is_prefix_of_source256'].sum())
    results['study']=STUDY;results['source_manifest_sha256']=sha(sealed)
    atomic_json(ROOT/'FINAL_FROZEN_R08_STATISTICS.json',results)
    print(json.dumps(results,indent=2))

    # Human audit is separate from the strict endpoint. Reviewers should not receive answer keys.
    blind=[];key=[]
    for c in CASES:
        for ph in PHASES:
            r=lookup[(c['id'],ph)]
            refid=sha_bytes(c['id']+'|'+ph)[:20]
            blind.append({'review_id':refid,'dataset':c['dataset'],'question':c['prompt'],
                          'response':r['output'],
                          'human_label':'','failure_reason':'','reviewer_notes':''})
            key.append({'review_id':refid,'target':c['target'],'primary_strict_label':r['grade']['label'],
                        'problem_id':c['id'],'phase':ph})
    pd.DataFrame(blind).sample(frac=1,random_state=2508).to_csv(ROOT/'R08_REVIEWER_A_BLINDED.csv',index=False)
    pd.DataFrame(blind).sample(frac=1,random_state=2509).to_csv(ROOT/'R08_REVIEWER_B_BLINDED.csv',index=False)
    pd.DataFrame(key).sort_values('review_id').to_csv(ROOT/'R08_SEALED_GOLD_ANSWERS.csv',index=False)
    print('Final outputs complete. Keep the sealed answer key away from reviewers.')
else:
    print('R08 is PARTIAL — do not make confirmatory claims; rerun notebook to resume saved trials.')
export_snapshot()
